<a href="https://colab.research.google.com/github/balasri03/Mini_project/blob/main/semevalTaskA_with_bert_bart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LSTM+BASELINE

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical

In [ ]:
train = pd.read_excel("train_data_semeval2016.xlsx")
test = pd.read_excel("SemEval2016-Task6-subtaskA-testdata-gold.xlsx")

print(train.head())

In [ ]:
train = train[['Tweet','Target','Stance']]
test = test[['Tweet','Target','Stance']]

In [ ]:
train['text'] = train['Tweet'] + " " + train['Target']
test['text'] = test['Tweet'] + " " + test['Target']

In [ ]:
le = LabelEncoder()

y_train = le.fit_transform(train['Stance'])
y_test = le.transform(test['Stance'])

y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

In [ ]:
vocab_size = 20000
max_len = 40

tokenizer = Tokenizer(num_words=vocab_size)

tokenizer.fit_on_texts(train['text'])

X_train = tokenizer.texts_to_sequences(train['text'])
X_test = tokenizer.texts_to_sequences(test['text'])

X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

In [ ]:
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import EarlyStopping

model_lstm = Sequential()

model_lstm.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=100,
        input_length=max_len
    )
)

model_lstm.add(LSTM(64))

model_lstm.add(Dropout(0.5))

model_lstm.add(Dense(3, activation='softmax'))

model_lstm.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)


In [ ]:
import time

train_start = time.time()

history = model_lstm.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop]
)

train_end = time.time()

train_time_hours = (train_end - train_start) / 3600

print("Training Time (hours):", train_time_hours)

In [ ]:
infer_start = time.time()

y_pred = model_lstm.predict(X_test)

infer_end = time.time()

infer_time = infer_end - infer_start

print("Inference Time (seconds):", infer_time)

In [ ]:
params = model_lstm.count_params() / 1e6

print("Parameters (M):", params)

In [ ]:
import numpy as np

y_pred_labels = np.argmax(y_pred, axis=1)
y_true_labels = np.argmax(y_test, axis=1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_true_labels, y_pred_labels)

precision = precision_score(y_true_labels, y_pred_labels, average="weighted")

recall = recall_score(y_true_labels, y_pred_labels, average="weighted")

f1 = f1_score(y_true_labels, y_pred_labels, average="weighted")

params = model_lstm.count_params() / 1e6

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("Training Time (h):", train_time_hours)
print("Inference Time (s):", infer_time)
print("Parameters (M):", params)

In [ ]:
pred = model_lstm.predict(X_test)

pred_labels = np.argmax(pred, axis=1)
true_labels = np.argmax(y_test, axis=1)

accuracy = accuracy_score(true_labels, pred_labels)

print("LSTM Baseline Accuracy:", accuracy)

In [ ]:
y_test_bin = label_binarize(true_labels, classes=[0,1,2])

fpr, tpr, _ = roc_curve(y_test_bin.ravel(), pred.ravel())

roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label="LSTM Baseline AUC=" + str(round(roc_auc,3)))

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")

plt.legend()

plt.show()

ONE HOT

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model_lstm_onehot = Sequential()

# OneHot style embedding (random initialized but trainable)
model_lstm_onehot.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=max_len,
        trainable=True
    )
)

# LSTM layer
model_lstm_onehot.add(
    LSTM(
        80,
        dropout=0.3,
        recurrent_dropout=0.3
    )
)

# Dropout to reduce overfitting
model_lstm_onehot.add(Dropout(0.4))

# Output layer
model_lstm_onehot.add(Dense(3, activation='softmax'))

# Compile
model_lstm_onehot.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)



In [ ]:
# from tensorflow.keras.callbacks import EarlyStopping
# early_stop = EarlyStopping(
#     monitor='val_loss',
#     patience=2,
#     restore_best_weights=True
# )

# history_onehot = model_lstm_onehot.fit(
#     X_train,
#     y_train,
#     epochs=10,
#     batch_size=32,
#     validation_split=0.1,
#     callbacks=[early_stop]
# )
import time
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# Training time start
train_start = time.time()

history_onehot = model_lstm_onehot.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop]
)

# Training time end
train_end = time.time()

train_time_hours = (train_end - train_start) / 3600

print("Training Time (hours):", train_time_hours)

In [ ]:
infer_start = time.time()

y_pred = model_lstm_onehot.predict(X_test)

infer_end = time.time()

infer_time = infer_end - infer_start

print("Inference Time (seconds):", infer_time)

In [ ]:
import numpy as np

y_pred_labels = np.argmax(y_pred, axis=1)
y_true_labels = np.argmax(y_test, axis=1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_true_labels, y_pred_labels)
precision = precision_score(y_true_labels, y_pred_labels, average="weighted")
recall = recall_score(y_true_labels, y_pred_labels, average="weighted")
f1 = f1_score(y_true_labels, y_pred_labels, average="weighted")

params = model_lstm_onehot.count_params() / 1e6

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("Training Time (h):", train_time_hours)
print("Inference Time (s):", infer_time)
print("Parameters (M):", params)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

pred2 = model_lstm_onehot.predict(X_test)

pred_labels2 = np.argmax(pred2, axis=1)
true_labels = np.argmax(y_test, axis=1)

accuracy2 = accuracy_score(true_labels, pred_labels2)
precision2 = precision_score(true_labels, pred_labels2, average='macro')
recall2 = recall_score(true_labels, pred_labels2, average='macro')
f12 = f1_score(true_labels, pred_labels2, average='macro')

print("LSTM OneHot Results")
print("Accuracy :", accuracy2)
print("Precision:", precision2)
print("Recall   :", recall2)
print("F1 Score :", f12)

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import numpy as np

# Binarize labels for multi-class ROC
y_test_bin = label_binarize(np.argmax(y_test, axis=1), classes=[0,1,2])

# Use model predictions
y_pred = model_lstm_onehot.predict(X_test)

# Compute ROC
fpr, tpr, _ = roc_curve(y_test_bin.ravel(), y_pred.ravel())

roc_auc = auc(fpr, tpr)

# Plot
plt.figure()

plt.plot(fpr, tpr, label="LSTM One-Hot (AUC = %0.3f)" % roc_auc)

plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve – LSTM One-Hot")

plt.legend()

plt.show()

LSTM+GLOVE

In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip

In [ ]:
import zipfile

with zipfile.ZipFile("glove.6B.zip", 'r') as zip_ref:
    zip_ref.extractall()

print("Extraction completed")

In [ ]:
embedding_dim = 100
embeddings_index = {}

with open("glove.6B.100d.txt", encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = vector

print("Total GloVe words loaded:", len(embeddings_index))

In [ ]:
word_index = tokenizer.word_index

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in word_index.items():

    if i >= vocab_size:
        continue

    embedding_vector = embeddings_index.get(word)

    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

In [ ]:
from tensorflow.keras.layers import SpatialDropout1D

model_lstm_glove = Sequential()

model_lstm_glove.add(
    Embedding(
        vocab_size,
        embedding_dim,
        weights=[embedding_matrix],
        input_length=max_len,
        trainable=False
    )
)

model_lstm_glove.add(SpatialDropout1D(0.3))

model_lstm_glove.add(LSTM(64))

model_lstm_glove.add(Dense(3, activation='softmax'))

model_lstm_glove.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
# history_glove = model_lstm_glove.fit(
#     X_train,
#     y_train,
#     epochs=8,
#     batch_size=32,
#     validation_split=0.1
# )
import time

train_start = time.time()

history_glove = model_lstm_glove.fit(
    X_train,
    y_train,
    epochs=8,
    batch_size=32,
    validation_split=0.1
)

train_end = time.time()

train_time_hours = (train_end - train_start) / 3600

print("Training Time (hours):", train_time_hours)

In [ ]:
infer_start = time.time()

y_pred = model_lstm_glove.predict(X_test)

infer_end = time.time()

infer_time = infer_end - infer_start

print("Inference Time (seconds):", infer_time)

In [ ]:
import numpy as np

y_pred_labels = np.argmax(y_pred, axis=1)
y_true_labels = np.argmax(y_test, axis=1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_true_labels, y_pred_labels)

precision = precision_score(y_true_labels, y_pred_labels, average="weighted")

recall = recall_score(y_true_labels, y_pred_labels, average="weighted")

f1 = f1_score(y_true_labels, y_pred_labels, average="weighted")

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

In [ ]:
params = model_lstm_glove.count_params() / 1e6

print("Parameters (M):", params)

In [ ]:
import pandas as pd

result = pd.DataFrame({
    "Dataset": ["SemEval-A"],
    "Model": ["LSTM + GloVe"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1-Score": [f1],
    "Train Time (h)": [train_time_hours],
    "Infer Time (s)": [infer_time],
    "Params (M)": [params]
})

print(result)

In [ ]:
pred3 = model_lstm_glove.predict(X_test)

pred_labels3 = np.argmax(pred3, axis=1)
true_labels = np.argmax(y_test, axis=1)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy3 = accuracy_score(true_labels, pred_labels3)
precision3 = precision_score(true_labels, pred_labels3, average='macro')
recall3 = recall_score(true_labels, pred_labels3, average='macro')
f13 = f1_score(true_labels, pred_labels3, average='macro')

print("LSTM + GloVe Results")
print("Accuracy :", accuracy3)
print("Precision:", precision3)
print("Recall   :", recall3)
print("F1 Score :", f13)

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import numpy as np

# Predictions from model
y_pred = model_lstm_glove.predict(X_test)

# True labels
true_labels = np.argmax(y_test, axis=1)

# Binarize labels
y_test_bin = label_binarize(true_labels, classes=[0,1,2])

# ROC
fpr, tpr, _ = roc_curve(y_test_bin.ravel(), y_pred.ravel())

roc_auc = auc(fpr, tpr)

# Plot
plt.figure()

plt.plot(fpr, tpr, label="LSTM + GloVe (AUC = %0.3f)" % roc_auc)

plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve – LSTM + GloVe")

plt.legend()

plt.show()

GLOVE+ATTENTION

In [ ]:
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.layers import GlobalAveragePooling1D
from tensorflow.keras.layers import Attention
from tensorflow.keras.models import Model

In [ ]:
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.layers import Bidirectional, GlobalMaxPooling1D
from tensorflow.keras.layers import Attention, Dropout
from tensorflow.keras.models import Model

input_layer = Input(shape=(max_len,))

embedding_layer = Embedding(
    vocab_size,
    embedding_dim,
    weights=[embedding_matrix],
    input_length=max_len,
    trainable=True
)(input_layer)

lstm_layer = Bidirectional(
    LSTM(64, return_sequences=True)
)(embedding_layer)

attention_layer = Attention()([lstm_layer, lstm_layer])

pooling_layer = GlobalMaxPooling1D()(attention_layer)

drop = Dropout(0.4)(pooling_layer)

output_layer = Dense(3, activation='softmax')(drop)

model_attention = Model(inputs=input_layer, outputs=output_layer)

model_attention.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model_attention.summary()

In [ ]:
# history_attention = model_attention.fit(
#     X_train,
#     y_train,
#     epochs=6,
#     batch_size=32,
#     validation_split=0.1
# )
import time

train_start = time.time()

history_attention = model_attention.fit(
    X_train,
    y_train,
    epochs=4,
    batch_size=32,
    validation_split=0.1
)

train_end = time.time()

train_time_hours = (train_end - train_start) / 3600

print("Training Time (hours):", train_time_hours)

In [ ]:
infer_start = time.time()

y_pred = model_attention.predict(X_test)

infer_end = time.time()

infer_time = infer_end - infer_start

print("Inference Time (seconds):", infer_time)

In [ ]:
import numpy as np

y_pred_labels = np.argmax(y_pred, axis=1)
y_true_labels = np.argmax(y_test, axis=1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_true_labels, y_pred_labels)

precision = precision_score(y_true_labels, y_pred_labels, average="weighted")

recall = recall_score(y_true_labels, y_pred_labels, average="weighted")

f1 = f1_score(y_true_labels, y_pred_labels, average="weighted")

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

In [ ]:
params = model_attention.count_params() / 1e6

print("Parameters (M):", params)

In [ ]:
import pandas as pd

result = pd.DataFrame({
    "Dataset": ["SemEval-A"],
    "Model": ["LSTM + GloVe + Attention"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1-Score": [f1],
    "Train Time (h)": [train_time_hours],
    "Infer Time (s)": [infer_time],
    "Params (M)": [params]
})

print(result)

In [ ]:
pred4 = model_attention.predict(X_test)

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np
import matplotlib.pyplot as plt

true_labels = np.argmax(y_test, axis=1)

y_test_bin = label_binarize(true_labels, classes=[0,1,2])

fpr, tpr, _ = roc_curve(y_test_bin.ravel(), pred4.ravel())

roc_auc = auc(fpr, tpr)

plt.figure()

plt.plot(fpr, tpr, label="LSTM + GloVe + Attention (AUC = %0.3f)" % roc_auc)

plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve – LSTM + GloVe + Attention")

plt.legend()

plt.show()

GRU BASELINE

In [ ]:
from tensorflow.keras.layers import GRU, SpatialDropout1D, Dropout
from tensorflow.keras.models import Sequential

model_gru_baseline = Sequential()

model_gru_baseline.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=100,
        input_length=max_len
    )
)

model_gru_baseline.add(SpatialDropout1D(0.3))

model_gru_baseline.add(GRU(64))

model_gru_baseline.add(Dropout(0.4))

model_gru_baseline.add(Dense(3, activation='softmax'))

model_gru_baseline.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
# from tensorflow.keras.callbacks import EarlyStopping

# early_stop = EarlyStopping(
#     monitor='val_loss',
#     patience=2,
#     restore_best_weights=True
# )

# history_gru = model_gru_baseline.fit(
#     X_train,
#     y_train,
#     epochs=10,
#     batch_size=32,
#     validation_split=0.1,
#     callbacks=[early_stop]
# )
import time

train_start = time.time()

history_attention = model_attention.fit(
    X_train,
    y_train,
    epochs=6,
    batch_size=32,
    validation_split=0.1
)

train_end = time.time()

train_time_hours = (train_end - train_start) / 3600

print("Training Time (hours):", train_time_hours)

In [ ]:
infer_start = time.time()

y_pred = model_attention.predict(X_test)

infer_end = time.time()

infer_time = infer_end - infer_start

print("Inference Time (seconds):", infer_time)

In [ ]:
import numpy as np

y_pred_labels = np.argmax(y_pred, axis=1)
y_true_labels = np.argmax(y_test, axis=1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_true_labels, y_pred_labels)

precision = precision_score(y_true_labels, y_pred_labels, average="weighted")

recall = recall_score(y_true_labels, y_pred_labels, average="weighted")

f1 = f1_score(y_true_labels, y_pred_labels, average="weighted")

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

In [ ]:
params = model_attention.count_params() / 1e6

print("Parameters (M):", params)

In [ ]:
import pandas as pd

result = pd.DataFrame({
    "Dataset": ["SemEval-A"],
    "Model": ["LSTM + GloVe + Attention"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1-Score": [f1],
    "Train Time (h)": [train_time_hours],
    "Infer Time (s)": [infer_time],
    "Params (M)": [params]
})

print(result)

In [ ]:
pred_gru = model_gru_baseline.predict(X_test)

pred_labels_gru = np.argmax(pred_gru, axis=1)
true_labels = np.argmax(y_test, axis=1)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy_gru = accuracy_score(true_labels, pred_labels_gru)
precision_gru = precision_score(true_labels, pred_labels_gru, average='macro')
recall_gru = recall_score(true_labels, pred_labels_gru, average='macro')
f1_gru = f1_score(true_labels, pred_labels_gru, average='macro')

print("GRU Baseline Results")
print("Accuracy :", accuracy_gru)
print("Precision:", precision_gru)
print("Recall   :", recall_gru)
print("F1 Score :", f1_gru)

In [ ]:
import numpy as np
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

# predictions
pred_gru = model_gru_baseline.predict(X_test)

# true labels
true_labels = np.argmax(y_test, axis=1)

# binarize
y_test_bin = label_binarize(true_labels, classes=[0,1,2])

# ROC
fpr, tpr, _ = roc_curve(y_test_bin.ravel(), pred_gru.ravel())

roc_auc = auc(fpr, tpr)

plt.figure()

plt.plot(fpr, tpr, label="GRU Baseline (AUC = %0.3f)" % roc_auc)

plt.plot([0,1],[0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve – GRU Baseline")

plt.legend()

plt.show()

GRU+ONEHOT

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout

model_gru_onehot = Sequential()

model_gru_onehot.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=100,
        input_length=max_len
    )
)

model_gru_onehot.add(GRU(80))

model_gru_onehot.add(Dropout(0.4))

model_gru_onehot.add(Dense(3, activation='softmax'))

model_gru_onehot.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model_gru_onehot.summary()

In [ ]:
# from tensorflow.keras.callbacks import EarlyStopping
# early_stop = EarlyStopping(
#     monitor='val_loss',
#     patience=2,
#     restore_best_weights=True
# )
# history_gru_onehot = model_gru_onehot.fit(
#     X_train,
#     y_train,
#     epochs=10,
#     batch_size=32,
#     validation_split=0.1,
#     callbacks=[early_stop]
# )
import time
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

train_start = time.time()

history_gru_onehot = model_gru_onehot.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop]
)

train_end = time.time()

train_time_hours = (train_end - train_start) / 3600

print("Training Time (hours):", train_time_hours)

In [ ]:
infer_start = time.time()

pred_gru_onehot = model_gru_onehot.predict(X_test)

infer_end = time.time()

infer_time = infer_end - infer_start

print("Inference Time (seconds):", infer_time)

In [ ]:
import numpy as np

y_pred_labels = np.argmax(pred_gru_onehot, axis=1)
y_true_labels = np.argmax(y_test, axis=1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_true_labels, y_pred_labels)
precision = precision_score(y_true_labels, y_pred_labels, average="weighted")
recall = recall_score(y_true_labels, y_pred_labels, average="weighted")
f1 = f1_score(y_true_labels, y_pred_labels, average="weighted")

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

In [ ]:
params = model_gru_onehot.count_params() / 1e6
print("Parameters (M):", params)

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

true_labels = np.argmax(y_test, axis=1)

y_test_bin = label_binarize(true_labels, classes=[0,1,2])

fpr, tpr, _ = roc_curve(y_test_bin.ravel(), pred_gru_onehot.ravel())

roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label="GRU One-Hot (AUC = %0.3f)" % roc_auc)
plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve – GRU One-Hot")
plt.legend()
plt.show()

In [ ]:
import pandas as pd

# create result row
row = {
    "Dataset": "SemEval-A",
    "Model": "GRU",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1,
    "Train Time (h)": train_time_hours,
    "Infer (s)": infer_time,
    "Params (M)": params
}

# if table already exists append row
try:
    results_table = results_table._append(row, ignore_index=True)
except:
    results_table = pd.DataFrame([row])

# display table
print(results_table)

GRU+GLOVE

In [ ]:
embedding_dim = 100
embeddings_index = {}

with open("glove.6B.100d.txt", encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = vector

print("Total words loaded:", len(embeddings_index))

In [ ]:
word_index = tokenizer.word_index

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in word_index.items():

    if i >= vocab_size:
        continue

    embedding_vector = embeddings_index.get(word)

    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

In [ ]:
# from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D
# from tensorflow.keras.models import Sequential

# model_lstm_glove = Sequential()

# model_lstm_glove.add(
#     Embedding(
#         vocab_size,
#         embedding_dim,
#         weights=[embedding_matrix],
#         input_length=max_len,
#         trainable=True
#     )
# )

# model_lstm_glove.add(SpatialDropout1D(0.3))

# model_lstm_glove.add(LSTM(64, dropout=0.3, recurrent_dropout=0.3))

# model_lstm_glove.add(Dense(3, activation="softmax"))

# model_lstm_glove.compile(
#     loss="categorical_crossentropy",
#     optimizer="adam",
#     metrics=["accuracy"]
# )

# model_lstm_glove.summary()
from tensorflow.keras.layers import Embedding, GRU, Dense, SpatialDropout1D
from tensorflow.keras.models import Sequential

model_gru_glove = Sequential()

model_gru_glove.add(
    Embedding(
        vocab_size,
        embedding_dim,
        weights=[embedding_matrix],
        input_length=max_len,
        trainable=True
    )
)

model_gru_glove.add(SpatialDropout1D(0.3))

model_gru_glove.add(GRU(64, dropout=0.3, recurrent_dropout=0.3))

model_gru_glove.add(Dense(3, activation="softmax"))

model_gru_glove.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model_gru_glove.summary()

In [ ]:
import time
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Early stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# -----------------------
# Training Time
# -----------------------
train_start = time.time()

history_gru_glove = model_gru_glove.fit(
    X_train,
    y_train,
    epochs=12,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop]
)

train_end = time.time()
train_time_hours = (train_end - train_start) / 3600


# -----------------------
# Inference Time
# -----------------------
infer_start = time.time()

pred_gru_glove = model_gru_glove.predict(X_test)

infer_end = time.time()
infer_time = infer_end - infer_start


# -----------------------
# Convert Labels
# -----------------------
y_pred_labels = np.argmax(pred_gru_glove, axis=1)
y_true_labels = np.argmax(y_test, axis=1)


# -----------------------
# Metrics
# -----------------------
accuracy = accuracy_score(y_true_labels, y_pred_labels)
precision = precision_score(y_true_labels, y_pred_labels, average="weighted")
recall = recall_score(y_true_labels, y_pred_labels, average="weighted")
f1 = f1_score(y_true_labels, y_pred_labels, average="weighted")


# -----------------------
# Model Parameters
# -----------------------
params = model_gru_glove.count_params() / 1e6


# -----------------------
# Print Results
# -----------------------
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("Train Time (h):", train_time_hours)
print("Infer Time (s):", infer_time)
print("Params (M):", params)

In [ ]:
pred_glove = model_gru_glove.predict(X_test)

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

# convert true labels
true_labels = np.argmax(y_test, axis=1)

# binarize labels
y_test_bin = label_binarize(true_labels, classes=[0,1,2])

# compute ROC
fpr, tpr, _ = roc_curve(y_test_bin.ravel(), pred_glove.ravel())

# compute AUC
roc_auc = auc(fpr, tpr)

print("AUC:", roc_auc)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()

plt.plot(fpr, tpr, label="GRU + GloVe (AUC = %0.3f)" % roc_auc)

plt.plot([0,1],[0,1],'--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve – GRU + GloVe")

plt.legend()

plt.show()

GRU+GLOVE+ATTENTION

In [ ]:
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.layers import Bidirectional, GlobalMaxPooling1D
from tensorflow.keras.layers import Attention, Dropout, SpatialDropout1D
from tensorflow.keras.models import Model

input_layer = Input(shape=(max_len,))

embedding_layer = Embedding(
    vocab_size,
    embedding_dim,
    weights=[embedding_matrix],
    input_length=max_len,
    trainable=True
)(input_layer)

drop1 = SpatialDropout1D(0.3)(embedding_layer)

bilstm = Bidirectional(
    LSTM(32, return_sequences=True)
)(drop1)

attention = Attention()([bilstm, bilstm])

pool = GlobalMaxPooling1D()(attention)

drop2 = Dropout(0.5)(pool)

output = Dense(3, activation='softmax')(drop2)

model_attention = Model(inputs=input_layer, outputs=output)

model_attention.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model_attention.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [ ]:
# history_attention = model_attention.fit(
#     X_train,
#     y_train,
#     epochs=12,
#     batch_size=32,
#     validation_split=0.1,
#     callbacks=[early_stop]
# )
import time
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# -----------------------
# Training Time
# -----------------------
train_start = time.time()

history_attention = model_attention.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=16,
    validation_split=0.1,
    callbacks=[early_stop]
)

train_end = time.time()
train_time_hours = (train_end - train_start) / 3600


# -----------------------
# Inference Time
# -----------------------
infer_start = time.time()

pred_attention = model_attention.predict(X_test)

infer_end = time.time()
infer_time = infer_end - infer_start


# -----------------------
# Labels
# -----------------------
y_pred_labels = np.argmax(pred_attention, axis=1)
y_true_labels = np.argmax(y_test, axis=1)


# -----------------------
# Metrics
# -----------------------
accuracy = accuracy_score(y_true_labels, y_pred_labels)
precision = precision_score(y_true_labels, y_pred_labels, average="weighted")
recall = recall_score(y_true_labels, y_pred_labels, average="weighted")
f1 = f1_score(y_true_labels, y_pred_labels, average="weighted")


# -----------------------
# Parameters
# -----------------------
params = model_attention.count_params() / 1e6


# -----------------------
# Print Results
# -----------------------
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("Train Time (h):", train_time_hours)
print("Infer Time (s):", infer_time)
print("Params (M):", params)

In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np

# Predict probabilities
pred_attention = model_attention.predict(X_test)

# Convert probabilities → class labels
pred_labels = np.argmax(pred_attention, axis=1)

# True labels
true_labels = np.argmax(y_test, axis=1)

# Calculate accuracy
accuracy = accuracy_score(true_labels, pred_labels)

print("Model Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(true_labels, pred_labels, average='macro')
recall = recall_score(true_labels, pred_labels, average='macro')
f1 = f1_score(true_labels, pred_labels, average='macro')

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

# Convert true labels to binary (multiclass ROC)
true_labels = np.argmax(y_test, axis=1)
y_test_bin = label_binarize(true_labels, classes=[0,1,2])

# Use model predictions (example: attention model)
pred_probs = model_attention.predict(X_test)

# Compute ROC
fpr, tpr, _ = roc_curve(y_test_bin.ravel(), pred_probs.ravel())

# Compute AUC
roc_auc = auc(fpr, tpr)

# Plot ROC
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label="Attention Model AUC = " + str(round(roc_auc,3)))
plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")

plt.legend()
plt.show()

In [ ]:
!pip install pandas numpy scikit-learn nltk torch torchtext tqdm

In [ ]:
import pandas as pd
import numpy as np
import re
import time
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

from torch.utils.data import Dataset, DataLoader
from collections import Counter
from tqdm import tqdm

In [ ]:
train_df = pd.read_excel("train_data_semeval2016.xlsx")
test_df = pd.read_excel("SemEval2016-Task6-subtaskA-testdata-gold.xlsx")

print(train_df.head())

In [ ]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"#", "", text)

    text = re.sub(r"[^a-zA-Z\s]", "", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text


train_df["Tweet"] = train_df["Tweet"].apply(clean_text)
test_df["Tweet"] = test_df["Tweet"].apply(clean_text)

In [ ]:
le = LabelEncoder()

train_df["label"] = le.fit_transform(train_df["Stance"])
test_df["label"] = le.transform(test_df["Stance"])

In [ ]:
def build_vocab(texts, max_vocab=20000):

    counter = Counter()

    for text in texts:
        counter.update(text.split())

    vocab = {word:i+2 for i,(word,_) in enumerate(counter.most_common(max_vocab))}

    vocab["<PAD>"] = 0
    vocab["<UNK>"] = 1

    return vocab


vocab = build_vocab(train_df["Tweet"])
vocab_size = len(vocab)

print("Vocabulary size:", vocab_size)

In [ ]:
max_len = 40

def encode_text(text):

    tokens = text.split()

    seq = [vocab.get(word,1) for word in tokens]

    if len(seq) < max_len:
        seq = seq + [0]*(max_len-len(seq))
    else:
        seq = seq[:max_len]

    return seq


train_df["seq"] = train_df["Tweet"].apply(encode_text)
test_df["seq"] = test_df["Tweet"].apply(encode_text)

In [ ]:
embedding_dim = 100

embedding_matrix = np.random.normal(scale=0.6, size=(vocab_size, embedding_dim))

with open("glove.6B.100d.txt", encoding="utf8") as f:

    for line in f:

        values = line.split()

        word = values[0]

        vector = np.asarray(values[1:], dtype="float32")

        if word in vocab:
            idx = vocab[word]
            embedding_matrix[idx] = vector

In [ ]:
class TweetDataset(Dataset):

    def __init__(self, df):

        self.X = df["seq"].tolist()
        self.y = df["label"].tolist()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):

        return torch.tensor(self.X[idx]), torch.tensor(self.y[idx])

In [ ]:
train_dataset = TweetDataset(train_df)
test_dataset = TweetDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
# class MiniLSTM(nn.Module):

#     def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):

#         super().__init__()

#         self.embedding = nn.Embedding(vocab_size, embedding_dim)

#         self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))

#         self.lstm = nn.LSTM(
#             embedding_dim,
#             hidden_dim,
#             num_layers=1,
#             batch_first=True
#         )

#         self.fc = nn.Linear(hidden_dim, num_classes)

#     def forward(self, x):

#         x = self.embedding(x)

#         out, (h,c) = self.lstm(x)

#         out = h[-1]

#         out = self.fc(out)

#         return out
class MiniLSTM(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):

        x = self.embedding(x)

        out, (h,c) = self.lstm(x)

        out = h[-1]

        out = self.dropout(out)

        out = self.fc(out)

        return out

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = torch.tensor(class_weights,dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MiniLSTM(
    vocab_size,
    embedding_dim=100,
    hidden_dim=128,
    num_classes=3
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def count_parameters(model):

    return sum(p.numel() for p in model.parameters() if p.requires_grad)

params = count_parameters(model) / 1e6

print("Parameters (M):", params)

In [ ]:
epochs = 16

train_start = time.time()

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X,y in train_loader:

        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(X)

        loss = criterion(outputs,y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss {total_loss/len(train_loader):.4f}")

train_time = (time.time() - train_start)/3600

In [ ]:
model.eval()

y_true = []
y_pred = []

infer_start = time.time()

with torch.no_grad():

    for X,y in test_loader:

        X = X.to(device)

        outputs = model(X)

        preds = torch.argmax(outputs,dim=1).cpu().numpy()

        y_pred.extend(preds)
        y_true.extend(y.numpy())

infer_time = time.time() - infer_start

In [ ]:
precision = precision_score(y_true, y_pred, average="weighted")
recall = recall_score(y_true, y_pred, average="weighted")
f1 = f1_score(y_true, y_pred, average="weighted")

In [ ]:
table = pd.DataFrame({

"Dataset":["SemEval-A"],

"Model":["Mini-LSTM + GloVe"],

"Accuracy":[acc],

"Precision":[precision],

"Recall":[recall],

"F1-Score":[f1],

"Train Time (h)":[train_time],

"Infer Time (s)":[infer_time],

"Params (M)":[params]

})

print(table)

minlstm+glove+attention

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import numpy as np

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

max_len = 100
embedding_dim = 100
hidden_dim = 128
dropout = 0.3
epochs = 18
lr = 0.0007
num_classes = 3

In [ ]:
class Attention(nn.Module):

    def __init__(self, hidden_dim):
        super().__init__()

        self.attn = nn.Linear(hidden_dim*2, 1)

    def forward(self, lstm_output):

        # lstm_output shape
        # (batch, seq_len, hidden_dim*2)

        scores = self.attn(lstm_output).squeeze(-1)

        weights = torch.softmax(scores, dim=1)

        context = torch.sum(lstm_output * weights.unsqueeze(-1), dim=1)

        return context

In [ ]:
class MiniBiLSTM_Attention(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):

        super().__init__()

        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))

        # allow embeddings to train (important)
        self.embedding.weight.requires_grad = True

        # BiLSTM
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.attention = Attention(hidden_dim)

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_dim*2, num_classes)

    def forward(self, x):

        x = self.embedding(x)

        lstm_out, _ = self.lstm(x)

        context = self.attention(lstm_out)

        context = self.dropout(context)

        out = self.fc(context)

        return out

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MiniBiLSTM_Attention(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_classes=num_classes
).to(device)

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=lr)

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

params = count_parameters(model)/1e6

print("Total Parameters (M):", params)

In [ ]:
train_start = time.time()

for epoch in range(epochs):

    model.train()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    for X,y in train_loader:

        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(X)

        loss = criterion(outputs,y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs,dim=1)

        total_correct += (preds==y).sum().item()

        total_samples += y.size(0)

    train_acc = total_correct/total_samples

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Loss {total_loss/len(train_loader):.4f} | "
        f"Train Acc {train_acc:.4f}"
    )

train_time = (time.time()-train_start)/3600

In [ ]:
model.eval()

y_true=[]
y_pred=[]

infer_start=time.time()

with torch.no_grad():

    for X,y in test_loader:

        X = X.to(device)

        outputs = model(X)

        preds = torch.argmax(outputs,dim=1).cpu().numpy()

        y_pred.extend(preds)
        y_true.extend(y.numpy())

infer_time = time.time()-infer_start

In [ ]:
acc = accuracy_score(y_true,y_pred)

precision = precision_score(y_true,y_pred,average="weighted")

recall = recall_score(y_true,y_pred,average="weighted")

f1 = f1_score(y_true,y_pred,average="weighted")

print("Accuracy:",acc)
print("Precision:",precision)
print("Recall:",recall)
print("F1:",f1)

In [ ]:
import pandas as pd

table = pd.DataFrame({

"Dataset":["SemEval-A"],

"Model":["Mini-BiLSTM + GloVe + Attention"],

"Accuracy":[acc],

"Precision":[precision],

"Recall":[recall],

"F1-Score":[f1],

"Train Time (h)":[train_time],

"Infer Time (s)":[infer_time],

"Params (M)":[params]

})

print(table)

mingru+glove embeddings

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import time

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

embedding_dim = 100
hidden_dim = 128
dropout = 0.3
epochs = 15
lr = 0.0005
num_classes = 3

In [ ]:
class MiniGRU(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):

        super().__init__()

        # GloVe embedding
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))
        self.embedding.weight.requires_grad = True

        # GRU
        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):

        x = self.embedding(x)

        out, h = self.gru(x)

        h = h[-1]

        h = self.dropout(h)

        out = self.fc(h)

        return out

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MiniGRU(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_classes=num_classes
).to(device)

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=lr)

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

params = count_parameters(model) / 1e6

print("Total Parameters (M):", params)

In [ ]:
train_start = time.time()

for epoch in range(epochs):

    model.train()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    for X, y in train_loader:

        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(X)

        loss = criterion(outputs, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        total_correct += (preds == y).sum().item()

        total_samples += y.size(0)

    train_acc = total_correct / total_samples

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Loss {total_loss/len(train_loader):.4f} | "
        f"Train Acc {train_acc:.4f}"
    )

train_time = (time.time() - train_start) / 3600

In [ ]:
model.eval()

y_true = []
y_pred = []

infer_start = time.time()

with torch.no_grad():

    for X, y in test_loader:

        X = X.to(device)

        outputs = model(X)

        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        y_pred.extend(preds)
        y_true.extend(y.numpy())

infer_time = time.time() - infer_start

In [ ]:
acc = accuracy_score(y_true, y_pred)

precision = precision_score(y_true, y_pred, average="weighted")

recall = recall_score(y_true, y_pred, average="weighted")

f1 = f1_score(y_true, y_pred, average="weighted")

print("Accuracy:", acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

In [ ]:
import pandas as pd

table = pd.DataFrame({

"Dataset":["SemEval-A"],

"Model":["Mini-GRU + GloVe"],

"Accuracy":[acc],

"Precision":[precision],

"Recall":[recall],

"F1-Score":[f1],

"Train Time (h)":[train_time],

"Infer Time (s)":[infer_time],

"Params (M)":[params]

})

print(table)

min gru+glove+attention

In [ ]:
import torch
import torch.nn as nn

class Attention(nn.Module):

    def __init__(self, hidden_dim):
        super().__init__()

        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, gru_output):

        # gru_output shape
        # (batch_size, seq_len, hidden_dim)

        scores = self.attn(gru_output).squeeze(-1)

        weights = torch.softmax(scores, dim=1)

        context = torch.sum(gru_output * weights.unsqueeze(-1), dim=1)

        return context

In [ ]:
class MiniGRU_Attention(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):

        super().__init__()

        # GloVe embedding
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))
        self.embedding.weight.requires_grad = True

        # GRU
        # self.gru = nn.GRU(
        #     embedding_dim,
        #     hidden_dim,
        #     num_layers=1,
        #     batch_first=True
        # )
        self.gru = nn.GRU(
    embedding_dim,
    hidden_dim,
    num_layers=1,
    batch_first=True,
    bidirectional=True
)

        # Attention
        # self.attention = Attention(hidden_dim)

        self.dropout = nn.Dropout(0.3)

        # self.fc = nn.Linear(hidden_dim, num_classes)
        self.attention = Attention(hidden_dim*2)
        self.fc = nn.Linear(hidden_dim*2, num_classes)

    def forward(self, x):

        x = self.embedding(x)

        gru_out, _ = self.gru(x)

        context = self.attention(gru_out)

        context = self.dropout(context)

        out = self.fc(context)

        return out

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MiniGRU_Attention(
    vocab_size=vocab_size,
    embedding_dim=100,
    hidden_dim=128,
    num_classes=3
).to(device)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = torch.tensor(class_weights,dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.0004)

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

params = count_parameters(model)/1e6

print("Params (M):", params)

In [ ]:
import time

train_start = time.time()

for epoch in range(15):

    model.train()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    for X,y in train_loader:

        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(X)

        loss = criterion(outputs,y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs,dim=1)

        total_correct += (preds==y).sum().item()

        total_samples += y.size(0)

    train_acc = total_correct/total_samples

    print(f"Epoch {epoch+1} | Loss {total_loss/len(train_loader):.4f} | Acc {train_acc:.4f}")

train_time = (time.time()-train_start)/3600

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model.eval()

y_true=[]
y_pred=[]

infer_start=time.time()

with torch.no_grad():

    for X,y in test_loader:

        X = X.to(device)

        outputs = model(X)

        preds = torch.argmax(outputs,dim=1).cpu().numpy()

        y_pred.extend(preds)
        y_true.extend(y.numpy())

infer_time = time.time()-infer_start

In [ ]:
acc = accuracy_score(y_true,y_pred)

precision = precision_score(y_true,y_pred,average="weighted")

recall = recall_score(y_true,y_pred,average="weighted")

f1 = f1_score(y_true,y_pred,average="weighted")

print("Accuracy:",acc)
print("Precision:",precision)
print("Recall:",recall)
print("F1:",f1)

In [ ]:
import pandas as pd

table = pd.DataFrame({

"Dataset":["SemEval-A"],

"Model":["Mini-GRU + GloVe + Attention"],

"Accuracy":[acc],

"Precision":[precision],

"Recall":[recall],

"F1-Score":[f1],

"Train Time (h)":[train_time],

"Infer Time (s)":[infer_time],

"Params (M)":[params]

})

print(table)

bert

In [ ]:
!pip install transformers

In [ ]:
import pandas as pd
import torch
import time

from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
train_df = pd.read_excel("train_data_semeval2016.xlsx")
test_df = pd.read_excel("SemEval2016-Task6-subtaskA-testdata-gold.xlsx")

In [ ]:
le = LabelEncoder()

train_df["label"] = le.fit_transform(train_df["Stance"])
test_df["label"] = le.transform(test_df["Stance"])

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

MAX_LEN = 100

In [ ]:
class StanceDataset(Dataset):

    def __init__(self, texts, labels):

        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        item = {k:v.squeeze(0) for k,v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx])

        return item

In [ ]:
train_dataset = StanceDataset(
    train_df["Tweet"].tolist(),
    train_df["label"].tolist()
)

test_dataset = StanceDataset(
    test_df["Tweet"].tolist(),
    test_df["label"].tolist()
)

train_loader = DataLoader(train_dataset,batch_size=16,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=16)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

model.to(device)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

In [ ]:
# epochs = 3

# train_start = time.time()

# for epoch in range(epochs):

#     model.train()

#     total_loss = 0

#     for batch in train_loader:

#         batch = {k:v.to(device) for k,v in batch.items()}

#         optimizer.zero_grad()

#         outputs = model(**batch)

#         loss = outputs.loss

#         loss.backward()

#         torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)

#         optimizer.step()

#         total_loss += loss.item()

#     print(f"Epoch {epoch+1} Loss {total_loss/len(train_loader):.4f}")

# train_time = (time.time()-train_start)/3600
# epochs = 3

# train_start = time.time()

# for epoch in range(epochs):

#     model.train()
#     total_loss = 0

#     for batch in train_loader:

#         batch = {k: v.to(device) for k, v in batch.items()}

#         optimizer.zero_grad()

#         outputs = model(**batch)
#         loss = outputs.loss

#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

#         optimizer.step()

#         total_loss += loss.item()

#     print(f"Epoch {epoch+1} Loss {total_loss/len(train_loader):.4f}")

# # ✅ Calculate training time
# train_time_seconds = time.time() - train_start
# train_time_minutes = train_time_seconds / 60
# train_time_hours = train_time_seconds / 3600

# # ✅ Print nicely
# print("\n===== Training Time =====")
# print(f"Training Time (seconds): {train_time_seconds:.4f}")
# print(f"Training Time (minutes): {train_time_minutes:.4f}")
# print(f"Training Time (hours): {train_time_hours:.4f}")
import time

epochs = 5   # better than 3

train_start = time.time()   # 🔥 start time

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch in train_loader:

        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()

        # 🔥 Forward pass
        outputs = model(**batch)
        logits = outputs.logits

        # 🔥 Weighted loss (for class imbalance)
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fn(logits, batch["labels"])

        # Backpropagation
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

# 🔥 END TIME
train_end = time.time()

# 🔥 CALCULATIONS
train_time_seconds = train_end - train_start
train_time_minutes = train_time_seconds / 60
train_time_hours = train_time_seconds / 3600

# 🔥 PRINT RESULTS
print("\n===== Training Time =====")
print(f"Training Time (seconds): {train_time_seconds:.2f}")
print(f"Training Time (minutes): {train_time_minutes:.2f}")
print(f"Training Time (hours): {train_time_hours:.4f}")




In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", use_fast=False)

In [ ]:
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model.eval()

y_true = []
y_pred = []

infer_start = time.time()

with torch.no_grad():

    for batch in test_loader:

        labels = batch["labels"]

        batch = {k:v.to(device) for k,v in batch.items()}

        outputs = model(**batch)

        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()

        y_pred.extend(preds)
        y_true.extend(labels.numpy())

infer_time = time.time() - infer_start

In [ ]:
acc = accuracy_score(y_true, y_pred)

precision = precision_score(y_true, y_pred, average="weighted")

recall = recall_score(y_true, y_pred, average="weighted")

f1 = f1_score(y_true, y_pred, average="weighted")

print("Accuracy:", acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

In [ ]:
params = sum(p.numel() for p in model.parameters()) / 1e6

table = pd.DataFrame({

"Dataset":["SemEval-A"],
"Model":["BERT"],
"Accuracy":[acc],
"Precision":[precision],
"Recall":[recall],
"F1-Score":[f1],
"Infer Time (s)":[infer_time],
"Params (M)":[params]

})

print(table)

In [ ]:
import torch
import numpy as np

model.eval()

all_probs = []
all_labels = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        probs = torch.softmax(logits, dim=1)

        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

pred_bert = np.array(all_probs)
y_test = np.array(all_labels)

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

# Convert labels to one-hot
y_test_bin = label_binarize(y_test, classes=[0,1,2])

# Compute ROC
fpr, tpr, _ = roc_curve(y_test_bin.ravel(), pred_bert.ravel())

roc_auc = auc(fpr, tpr)

print("AUC:", roc_auc)

In [ ]:
plt.figure()

plt.plot(fpr, tpr, label="BERT (AUC = %0.3f)" % roc_auc)

plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve – BERT")

plt.legend()

plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:\n", cm)

# Get class names from LabelEncoder
labels = le.classes_

# Plot heatmap
plt.figure(figsize=(6,5))
sns.heatmap(cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=labels,
            yticklabels=labels)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - BERT")

plt.show()

bart

In [ ]:
!pip install transformers

In [ ]:
import pandas as pd
import torch
import time

from transformers import BartTokenizer, BartForSequenceClassification
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
train_df = pd.read_excel("train_data_semeval2016.xlsx")
test_df = pd.read_excel("SemEval2016-Task6-subtaskA-testdata-gold.xlsx")

In [ ]:
le = LabelEncoder()

train_df["label"] = le.fit_transform(train_df["Stance"])
test_df["label"] = le.transform(test_df["Stance"])

In [ ]:
tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")

MAX_LEN = 100

In [ ]:
class StanceDataset(Dataset):

    def __init__(self,texts,labels):

        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self,idx):

        encoding = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        item = {k:v.squeeze(0) for k,v in encoding.items()}

        item["labels"] = torch.tensor(self.labels[idx])

        return item

In [ ]:
train_dataset = StanceDataset(
    train_df["Tweet"].tolist(),
    train_df["label"].tolist()
)

test_dataset = StanceDataset(
    test_df["Tweet"].tolist(),
    test_df["label"].tolist()
)

train_loader = DataLoader(train_dataset,batch_size=16,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=16)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BartForSequenceClassification.from_pretrained(
    "facebook/bart-base",
    num_labels=3
)

model.to(device)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [ ]:
epochs = 4

train_start = time.time()

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for batch in train_loader:

        batch = {k:v.to(device) for k,v in batch.items()}

        optimizer.zero_grad()

        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss {total_loss/len(train_loader):.4f}")

train_time = (time.time()-train_start)/3600

In [ ]:
model.eval()

y_true=[]
y_pred=[]

infer_start=time.time()

with torch.no_grad():

    for batch in test_loader:

        labels=batch["labels"]

        batch={k:v.to(device) for k,v in batch.items()}

        outputs=model(**batch)

        preds=torch.argmax(outputs.logits,dim=1).cpu().numpy()

        y_pred.extend(preds)
        y_true.extend(labels.numpy())

infer_time=time.time()-infer_start

In [ ]:
acc = accuracy_score(y_true,y_pred)

precision = precision_score(y_true,y_pred,average="weighted")

recall = recall_score(y_true,y_pred,average="weighted")

f1 = f1_score(y_true,y_pred,average="weighted")

print("Accuracy:",acc)
print("Precision:",precision)
print("Recall:",recall)
print("F1:",f1)

In [ ]:
params = sum(p.numel() for p in model.parameters())/1e6

table = pd.DataFrame({

"Dataset":["SemEval-A"],
"Model":["BART"],
"Accuracy":[acc],
"Precision":[precision],
"Recall":[recall],
"F1-Score":[f1],
"Train Time (h)":[train_time],
"Infer Time (s)":[infer_time],


"Params (M)":[params]

})

print(table)

In [ ]:
table.to_csv("bart_results.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot heatmap
plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',       # light color theme
    cbar=True,
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - BART')
plt.show()

bert

In [ ]:
!pip install transformers

In [ ]:
import pandas as pd
import torch
import numpy as np
import time

from transformers import BertTokenizer, BertForSequenceClassification
from transformers import get_linear_schedule_with_warmup

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
train_df = pd.read_excel("train_data_semeval2016.xlsx")
test_df = pd.read_excel("SemEval2016-Task6-subtaskA-testdata-gold.xlsx")

print(train_df.head())

In [ ]:
le = LabelEncoder()

train_df["label"] = le.fit_transform(train_df["Stance"])
test_df["label"] = le.transform(test_df["Stance"])

print("Classes:", le.classes_)  # must be AGAINST, FAVOR, NONE

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
MAX_LEN = 128

In [ ]:
class StanceDataset(Dataset):

    def __init__(self, texts, targets, labels):
        self.texts = texts
        self.targets = targets
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        # 🔥 TARGET + TEXT (very important)
        text = "Target: " + str(self.targets[idx]) + " Text: " + str(self.texts[idx])

        encoding = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        item = {k:v.squeeze(0) for k,v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx])

        return item

In [ ]:
train_dataset = StanceDataset(
    train_df["Tweet"].tolist(),
    train_df["Target"].tolist(),
    train_df["label"].tolist()
)

test_dataset = StanceDataset(
    test_df["Tweet"].tolist(),
    test_df["Target"].tolist(),
    test_df["label"].tolist()
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

In [ ]:
epochs = 6

total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

In [ ]:
train_start = time.time()

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch in train_loader:

        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()

        outputs = model(**batch)
        logits = outputs.logits

        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fn(logits, batch["labels"])

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} Loss: {total_loss/len(train_loader):.4f}")

train_time = time.time() - train_start
print("\nTraining Time:", train_time)